Now that I have pushed some collections, I can use them in code to do my 
analysis, fully delegating I/O management to hippo! Most importantly, it does not
matter if I am on a different machine: hippo gaurantees portability and reproducibility.

In [ ]:
from henry import Henry
from henry.collection import CollectionInstance
import hippometa as hm 

from pixell import enmap, enplot, curvedsky
import numpy as np
from matplotlib import pyplot as plt

client = Henry()

I will demo a simple approximate power spectrum for pa6 f150. So I probably want the 
array-major collection. Let's grab it:
* Still need to know its id
* This time, I do `realize_sources=True`. After doing this once (or manually caching the collection in advance), hippo will know not to re-download products!

In [ ]:
mycol = client.pull_collection('6ad1be78b4c52fc5e10d4b4d', realize_sources=True, pull_children=True)

# put into a dictionary so I can lookup products by their metadata, written by
# Claude
def iter_products(collection):
    """Yield every product in a collection tree, depth-first."""
    for item in collection:
        if isinstance(item, CollectionInstance):
            yield from iter_products(item)
        else:
            yield item

def index_products(collection, keys):
    """Map tuple(metadata.<k> for k in keys) -> product. Missing fields become None."""
    index = {}
    for prod in iter_products(collection):
        index_key = tuple(getattr(prod.metadata, k, None) for k in keys)
        if index_key in index and index[index_key].product_id != prod.product_id:
            raise KeyError(f'Duplicate index key {index_key}: {index[index_key].name}, {prod.name}')
        index[index_key] = prod
    return index

keys = ('metadata_type', 'split') #  I know it's pa6_f150
remote_products_by_metadata = index_products(mycol, keys)

Ignoring my mess-up with the metadata of these windows (they only look different
than the maps in their metdata because splits is `None`, this is quite fragile),
I can now write a script that interacts solely with this dictionary!

This is very basic power spectra, that is besides the point...

In [ ]:
lmax = 2000

w = enmap.read_map(str(remote_products_by_metadata['mapset', None]['map'].path))
l, bl = np.loadtxt(remote_products_by_metadata['beam', None]['data'].path, usecols=(0, 1)).T

l = l[:lmax+1]
bl = bl[:lmax + 1]
bl /= bl[0]

# get the 4 splits! sel=0 just loads the intensity component for simplicity
imaps = [enmap.read_map(str(remote_products_by_metadata['mapset', i]['source_free'].path), sel=0) for i in range(4)]

# get my "pseudospectrum to powerspectrum" operator
w2 = np.sum(w**2 * w.pixsizemap()) / 4 / np.pi 
fac = l * (l + 1) / 2 / np.pi / bl**2 / w2

# get my alms
alms = [curvedsky.map2alm(w * imaps[i], lmax=lmax) for i in range(4)]

# get a cl from the average of split crosses
cl = 0 
count = 0
for i in range(4):
    for j in range(i + 1, 4): # no autos!
        cl += curvedsky.alm2cl(alms[i], alms[j])
        count += 1
cl /= count
cl *= fac

# what do we got
plt.plot(l, cl)
plt.ylabel('Dl_TT')
plt.title('pa6_f150 power spectrum')